In [ ]:
!pip install transformers
!pip install huggingface_hub
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `projectMagang` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
# load data
df = pd.read_csv('/content/dataset_sample_labelled.csv')
labels = ['pelayanan', 'fasilitas']

In [ ]:
# ubah jenis data di pelayanan dan fasilitas menjadi integer
df['pelayanan'] = df['pelayanan'].astype(int)
df['fasilitas'] = df['fasilitas'].astype(int)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      800 non-null    object
 1   stars      800 non-null    int64 
 2   text       796 non-null    object
 3   pelayanan  800 non-null    int64 
 4   fasilitas  800 non-null    int64 
dtypes: int64(3), object(2)
memory usage: 31.4+ KB


In [ ]:
# tokenizer
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# model
model = BertForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p1", num_labels = len(labels),
                                                      problem_type ="multi_label_classification")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].fillna('').tolist(), # Fill missing text values with empty string
    df[labels].values,
    test_size = 0.3,
    random_state = 42
)

In [ ]:
# Dataset Class
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# dataset
train_dataset = ReviewDataset(train_texts, train_labels, tokenizer)
val_dataset = ReviewDataset(val_texts, val_labels, tokenizer)

In [ ]:
# training args
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch", # Corrected argument name
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [ ]:
# Evaluation metric
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(pred):
    logits, labels = pred
    preds = torch.sigmoid(torch.tensor(logits)).numpy() > 0.5
    labels = labels.astype(int)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='micro')
    }

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: willywonka99 (willywonka99-sebelas-maret-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.309800,0.237199,0.833333,0.938346
2,0.222300,0.260403,0.783333,0.921481
3,0.152300,0.428712,0.762500,0.900787
4,0.103000,0.364866,0.820833,0.931921
5,0.057000,0.398979,0.812500,0.928463


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


TrainOutput(global_step=350, training_loss=0.18056746704237803, metrics={'train_runtime': 503.7802, 'train_samples_per_second': 5.558, 'train_steps_per_second': 0.695, 'total_flos': 502172115816000.0, 'train_loss': 0.18056746704237803, 'epoch': 5.0})

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add t

In [ ]:
from huggingface_hub import upload_folder

repo_id = "willywonka19/indobert-classification-rs-3"
model_dir = "/content/results/checkpoint-350"

upload_folder(
    repo_id=repo_id,
    folder_path=model_dir,
    commit_message="Upload fine-tuned IndoBERT checkpoint"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...esults/checkpoint-350/rng_state.pth:  78%|#######7  | 11.0kB / 14.2kB            

  ...results/checkpoint-350/scheduler.pt: 100%|##########| 1.06kB / 1.06kB            

  ...results/checkpoint-350/optimizer.pt:   0%|          | 19.6kB /  996MB            

  ...ts/checkpoint-350/model.safetensors:   1%|          | 3.87MB /  498MB            

  ...ts/checkpoint-350/training_args.bin:  16%|#5        |   845B / 5.30kB            

CommitInfo(commit_url='https://huggingface.co/willywonka19/indobert-classification-rs-3/commit/b29d3b6955d0344b3d35774f8a767574b3fd9717', commit_message='Upload fine-tuned IndoBERT checkpoint', commit_description='', oid='b29d3b6955d0344b3d35774f8a767574b3fd9717', pr_url=None, repo_url=RepoUrl('https://huggingface.co/willywonka19/indobert-classification-rs-3', endpoint='https://huggingface.co', repo_type='model', repo_id='willywonka19/indobert-classification-rs-3'), pr_revision=None, pr_num=None)